# AquaSmart — Comparaison des Stratégies d'Apprentissage Fédéré (Federated Learning)

Ce notebook compare **six approches** d'entraînement sur des données piscicoles **non-IID** réparties sur plusieurs bassins (ponds), puis sélectionne la meilleure stratégie fédérée à déployer dans l'application AquaSmart.

| # | Approche | Rôle | Confidentialité |
|---|----------|------|------------------|
| 1 | **Centralisé** (toutes les données regroupées) | Borne supérieure | ❌ aucune |
| 2 | **Local-only** (chaque bassin seul) | Borne inférieure | ✅ totale |
| 3 | **FedAvg** | FL de référence | ✅ |
| 4 | **FedProx** (terme proximal μ) | hétérogénéité non-IID | ✅ |
| 5 | **FedMedian** (médiane coordonnée) | robustesse aux outliers | ✅ |
| 6 | **FedAdam** (optimiseur serveur adaptatif) | convergence accélérée | ✅ |

Tous les modèles locaux sont des régressions linéaires régularisées (Ridge) entraînées par descente de gradient, ce qui rend les stratégies **directement comparables**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size']  = 11
PRIMARY = '#1565C0'
np.random.seed(42)

# ── Global hyper-parameters ──
N_PONDS       = 8       # nombre de bassins (clients)
N_ROUNDS      = 25      # rondes de communication
LOCAL_EPOCHS  = 8       # epochs locales par ronde
LEARNING_RATE = 0.05    # lr local (descente de gradient)
ALPHA         = 0.01    # regularisation Ridge (L2) du modele lineaire
print('Config OK — %d bassins, %d rondes' % (N_PONDS, N_ROUNDS))

## 1. Ingénierie des features et génération des données non-IID

Chaque bassin possède un **biais local** (température, oxygène, échelle de population) : les distributions diffèrent d'un bassin à l'autre, ce qui reproduit la réalité *non-IID* du Federated Learning. Aucune donnée n'est partagée entre bassins.

In [ ]:
def engineer(temp, do, ph, turb, ammo, nit, fw):
    """21 features derivees (miroir de ml_service._engineer)."""
    thermal_stress = max(0, 20 - temp) + max(0, temp - 28)
    do_deficit     = max(0, 5.0 - do)
    do_ok          = 1.0 if do >= 5.0 else 0.0
    ph_dev         = abs(ph - 7.0)
    wqi            = ((100*(1-min(thermal_stress/10,1))) + (min(do,8)/8*100) + (100-min(ph_dev/3,1)*100))/3
    env_stress     = thermal_stress + do_deficit*2 + ph_dev + turb/30 + ammo/0.02
    return [
        temp, do, ph, turb, ammo, nit, fw,                 # 7 bruts
        thermal_stress, do_deficit, do_ok, ph_dev, wqi,    # 5 derives
        env_stress,                                        # 1 composite
        temp*do, ph*do, temp*temp, do*do,                  # 4 interactions
        temp*ammo, fw*0.03, turb*ammo, nit*ph,             # 4 interactions
    ]

N_FEATURES = len(engineer(24,7,7.2,12,0.008,3,80))
print('N_FEATURES =', N_FEATURES)

def generate_pond_data(pond_id, n=400):
    # Fort biais local => heterogeneite NON-IID marquee entre bassins
    rng = np.random.default_rng(pond_id*137 + 42)
    temp_bias = rng.uniform(-4, 4); do_bias = rng.uniform(-1.5, 1.5); pop_scale = rng.uniform(0.5, 1.8)
    temp = np.clip(rng.normal(24+temp_bias, 2.0, n), 10, 38)
    do   = np.clip(rng.normal(7.0+do_bias, 1.0, n), 1, 12)
    ph   = np.clip(rng.normal(7.2, 0.4, n), 5, 10)
    turb = np.clip(rng.exponential(12, n), 0, 80)
    ammo = np.clip(rng.exponential(0.008, n), 0, 0.1)
    nit  = np.clip(rng.exponential(3, n), 0, 20)
    fw   = np.clip(rng.normal(80*pop_scale, 30, n), 5, 500)
    rows, targets = [], []
    for i in range(n):
        rows.append(engineer(temp[i], do[i], ph[i], turb[i], ammo[i], nit[i], fw[i]))
        base = fw[i]*0.03
        tf = 1.1 if 22<=temp[i]<=26 else (0.5 if temp[i]<18 else 0.8)
        df = 1.05 if do[i]>=7 else (0.5 if do[i]>=3 else 0.0)
        pf = 1.0 if 6.8<=ph[i]<=8.2 else 0.6
        targets.append(max(0.0, base*tf*df*pf + rng.normal(0, 0.4)))
    return np.array(rows, dtype=np.float64), np.array(targets, dtype=np.float64)

In [ ]:
# ── Build per-client train + a pooled GLOBAL test set ──
# Donnees LIMITEES par bassin (60-120) : chaque bassin seul generalise mal,
# ce qui revele l'interet du FL (agregation = effet d'un dataset elargi).
# Un scaler GLOBAL ajuste sur le pool d'entrainement est partage pour que
# toutes les approches soient evaluees sur la meme echelle (comparaison equitable).
raw_clients = []; Xtr_raw_all, ytr_all = [], []
Xg_te_raw, yg_te = [], []
rng = np.random.default_rng(7)
for pid in range(1, N_PONDS+1):
    n = int(rng.integers(60, 120))
    Xraw, y = generate_pond_data(pid, n)
    cut = int(0.8*n)
    raw_clients.append(dict(id=pid, Xtr_raw=Xraw[:cut], ytr=y[:cut], n=cut))
    Xtr_raw_all.append(Xraw[:cut]); ytr_all.append(y[:cut])
    Xg_te_raw.append(Xraw[cut:]);   yg_te.append(y[cut:])
Xtr_raw_all = np.vstack(Xtr_raw_all); ytr_all = np.concatenate(ytr_all)
Xg_te_raw   = np.vstack(Xg_te_raw);   yg_te   = np.concatenate(yg_te)

GLOBAL_SCALER = RobustScaler().fit(Xtr_raw_all)
clients = [dict(id=c['id'], Xtr=GLOBAL_SCALER.transform(c['Xtr_raw']), ytr=c['ytr'], n=c['n'])
           for c in raw_clients]
Xg_te = GLOBAL_SCALER.transform(Xg_te_raw)
print('Bassins:', N_PONDS, '| Test global:', Xg_te.shape, '| echantillons train totaux:', sum(c['n'] for c in clients))

## 2. Briques communes (modèle linéaire + métriques)

Un modèle = vecteur de paramètres `p = [w (21), b (1)]`. La descente de gradient locale minimise l'erreur quadratique moyenne avec régularisation Ridge.

In [ ]:
def predict(p, X):           return X @ p[:-1] + p[-1]
def metrics(p, X, y):
    pr = predict(p, X)
    return dict(rmse=float(np.sqrt(mean_squared_error(y, pr))),
                mae =float(mean_absolute_error(y, pr)),
                r2  =float(r2_score(y, pr)))

def local_train(global_p, Xtr, ytr, mu=0.0, dp_noise=0.0):
    """GD local; mu>0 => terme proximal FedProx. Retourne params mis a jour."""
    p = global_p.copy(); w, b = p[:-1].copy(), p[-1]
    gw0 = global_p[:-1]  # ancre proximale
    n = len(ytr); batch = max(16, n//8)
    for _ in range(LOCAL_EPOCHS):
        idx = np.random.permutation(n)
        for s in range(0, n, batch):
            ib = idx[s:s+batch]; Xb, yb = Xtr[ib], ytr[ib]
            err = Xb @ w + b - yb
            gw = (2/len(ib))*(Xb.T @ err) + 2*ALPHA*w + mu*(w - gw0)
            gb = (2/len(ib))*err.sum()
            if dp_noise > 0:
                gw = gw + np.random.normal(0, dp_noise, gw.shape)
                gb = gb + float(np.random.normal(0, dp_noise))
            w -= LEARNING_RATE*gw; b -= LEARNING_RATE*gb
    return np.append(w, b)

## 3. Bornes de référence : Centralisé et Local-only

In [ ]:
# ── (1) CENTRALISE : toutes les donnees regroupees (borne superieure, sans confidentialite) ──
# Solution Ridge en forme fermee sur le pool complet => veritable borne superieure.
from sklearn.linear_model import Ridge
Xc = np.vstack([c['Xtr'] for c in clients]); yc = np.concatenate([c['ytr'] for c in clients])
_ridge = Ridge(alpha=ALPHA, fit_intercept=True).fit(Xc, yc)
p_central = np.append(_ridge.coef_.flatten(), _ridge.intercept_)
central = metrics(p_central, Xg_te, yg_te)
print('Centralise :', {k: round(v,4) for k,v in central.items()})

# ── (2) LOCAL-ONLY : chaque bassin entraine seul, evalue sur le test GLOBAL (borne inferieure) ──
local_scores = []
for c in clients:
    rl = Ridge(alpha=ALPHA, fit_intercept=True).fit(c['Xtr'], c['ytr'])
    pl = np.append(rl.coef_.flatten(), rl.intercept_)
    local_scores.append(metrics(pl, Xg_te, yg_te))
local_only = {k: float(np.mean([s[k] for s in local_scores])) for k in ['rmse','mae','r2']}
print('Local-only :', {k: round(v,4) for k,v in local_only.items()})

## 4. Les quatre stratégies fédérées

Chaque stratégie partage la même boucle : à chaque ronde, les bassins entraînent localement puis le serveur **agrège** les paramètres. Seule l'agrégation (et, pour FedProx, le terme local) change.

In [ ]:
def run_federated(strategy, n_rounds=N_ROUNDS, mu=0.1, server_lr=0.05):
    """strategy in {'fedavg','fedprox','fedmedian','fedadam'}. Retourne (params, history)."""
    p = np.zeros(N_FEATURES+1)
    m_adam = np.zeros_like(p); v_adam = np.zeros_like(p)  # etat FedAdam
    b1, b2, tau = 0.9, 0.999, 1e-3
    history = []
    for rnd in range(1, n_rounds+1):
        updates, ns = [], []
        for c in clients:
            mu_c = mu if strategy == 'fedprox' else 0.0
            upd = local_train(p, c['Xtr'], c['ytr'], mu=mu_c)
            updates.append(upd); ns.append(c['n'])
        updates = np.array(updates); ns = np.array(ns, dtype=float)
        if strategy in ('fedavg', 'fedprox'):
            new_p = (ns[:,None]*updates).sum(0) / ns.sum()          # FedAvg pondere
        elif strategy == 'fedmedian':
            new_p = np.median(updates, axis=0)                      # mediane coordonnee
        elif strategy == 'fedadam':
            avg   = (ns[:,None]*updates).sum(0) / ns.sum()
            delta = avg - p                                         # pseudo-gradient serveur
            m_adam = b1*m_adam + (1-b1)*delta
            v_adam = b2*v_adam + (1-b2)*delta**2
            mhat = m_adam/(1-b1**rnd); vhat = v_adam/(1-b2**rnd)    # correction de biais
            new_p = p + server_lr * mhat/(np.sqrt(vhat)+tau)
        else:
            raise ValueError(strategy)
        p = new_p
        mt = metrics(p, Xg_te, yg_te)
        history.append(dict(round=rnd, **mt))
    return p, pd.DataFrame(history)

fl_results = {}
for strat in ['fedavg', 'fedprox', 'fedmedian', 'fedadam']:
    pfin, hist = run_federated(strat)
    fl_results[strat] = dict(params=pfin, history=hist, final=hist.iloc[-1][['rmse','mae','r2']].to_dict())
    print('%-10s -> RMSE=%.4f  MAE=%.4f  R2=%.4f' % (strat, *[fl_results[strat]['final'][k] for k in ['rmse','mae','r2']]))

## 5. Tableau comparatif des performances

In [ ]:
rows = [
    ('Centralisé (idéal)',   central['rmse'],    central['mae'],    central['r2'],    'Aucune'),
    ('Local-only (isolé)',   local_only['rmse'], local_only['mae'], local_only['r2'], 'Totale'),
    ('FedAvg',     fl_results['fedavg']['final']['rmse'],    fl_results['fedavg']['final']['mae'],    fl_results['fedavg']['final']['r2'],    'Totale'),
    ('FedProx',    fl_results['fedprox']['final']['rmse'],   fl_results['fedprox']['final']['mae'],   fl_results['fedprox']['final']['r2'],   'Totale'),
    ('FedMedian',  fl_results['fedmedian']['final']['rmse'], fl_results['fedmedian']['final']['mae'], fl_results['fedmedian']['final']['r2'], 'Totale'),
    ('FedAdam',    fl_results['fedadam']['final']['rmse'],   fl_results['fedadam']['final']['mae'],   fl_results['fedadam']['final']['r2'],   'Totale'),
]
comp = pd.DataFrame(rows, columns=['Approche','RMSE','MAE','R²','Confidentialité'])
comp[['RMSE','MAE','R²']] = comp[['RMSE','MAE','R²']].round(4)

# Meilleure stratégie FÉDÉRÉE (RMSE minimal parmi les 4 FL)
fl_only = comp[comp['Approche'].isin(['FedAvg','FedProx','FedMedian','FedAdam'])]
best = fl_only.loc[fl_only['RMSE'].idxmin(), 'Approche']
comp['Meilleur FL'] = np.where(comp['Approche']==best, '★', '')
display(comp)
print('\n>>> Meilleure stratégie fédérée :', best)

## 6. Visualisation — courbes de convergence

In [ ]:
colors = {'fedavg':'#1565C0','fedprox':'#2E7D32','fedmedian':'#E65100','fedadam':'#6A1B9A'}
labels = {'fedavg':'FedAvg','fedprox':'FedProx','fedmedian':'FedMedian','fedadam':'FedAdam'}
fig, ax = plt.subplots(1, 2, figsize=(14,5))
for s, r in fl_results.items():
    ax[0].plot(r['history']['round'], r['history']['rmse'], marker='o', ms=4, color=colors[s], label=labels[s])
    ax[1].plot(r['history']['round'], r['history']['r2'],   marker='o', ms=4, color=colors[s], label=labels[s])
ax[0].axhline(central['rmse'], ls='--', c='grey', label='Centralisé (idéal)')
ax[0].axhline(local_only['rmse'], ls=':', c='red', label='Local-only')
ax[0].set_title('Convergence — RMSE global par ronde', fontweight='bold')
ax[0].set_xlabel('Ronde de communication'); ax[0].set_ylabel('RMSE'); ax[0].legend(fontsize=9); ax[0].grid(alpha=.3)
ax[1].axhline(central['r2'], ls='--', c='grey', label='Centralisé (idéal)')
ax[1].set_title('Convergence — R² global par ronde', fontweight='bold')
ax[1].set_xlabel('Ronde de communication'); ax[1].set_ylabel('R²'); ax[1].legend(fontsize=9); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig('fl_convergence.png', dpi=130, bbox_inches='tight'); plt.show()

## 7. Visualisation — barres des performances finales

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14,5))
order = comp['Approche'].tolist()
bar_c = ['#9E9E9E','#EF5350','#1565C0','#2E7D32','#E65100','#6A1B9A']
ax[0].barh(order, comp['RMSE'], color=bar_c); ax[0].invert_yaxis()
ax[0].set_title('RMSE finale (plus bas = mieux)', fontweight='bold'); ax[0].set_xlabel('RMSE')
for i,v in enumerate(comp['RMSE']): ax[0].text(v, i, f' {v:.3f}', va='center', fontsize=9)
ax[1].barh(order, comp['R²'], color=bar_c); ax[1].invert_yaxis()
ax[1].set_title('R² finale (plus haut = mieux)', fontweight='bold'); ax[1].set_xlabel('R²')
for i,v in enumerate(comp['R²']): ax[1].text(v, i, f' {v:.3f}', va='center', fontsize=9)
plt.tight_layout(); plt.savefig('fl_comparison_bars.png', dpi=130, bbox_inches='tight'); plt.show()

## 8. Conclusion

Exécutez la cellule ci-dessous pour générer le verdict automatique à recopier dans le rapport.

In [ ]:
gap = (local_only['rmse'] - fl_results[best.lower()]['final']['rmse'])
print('='*64)
print('VERDICT'.center(64))
print('='*64)
print(f"Meilleure stratégie fédérée : {best}")
print(f"  RMSE = {fl_results[best.lower()]['final']['rmse']:.4f}")
print(f"  R²   = {fl_results[best.lower()]['final']['r2']:.4f}")
print(f"\nCentralisé (idéal, sans privacy) : RMSE = {central['rmse']:.4f}")
print(f"Local-only (isolé, lower bound)  : RMSE = {local_only['rmse']:.4f}")
print(f"\n=> {best} récupère ~{gap:.3f} RMSE par rapport à l'isolement,")
print(f"   tout en préservant totalement la confidentialité des bassins.")
print('='*64)
comp.to_csv('fl_comparison_table.csv', index=False)
print('Tableau exporté -> fl_comparison_table.csv')